# REST: Spheroid Tracking

## 1. Environment and input paths

Use an environment with the packages listed in `pyproject.toml`. The installation command is optional when they are already available. The trained checkpoint and input image sequence are external files and must be specified below. Frames are expected as image files in one directory.

In [ ]:
%pip install ultralytics==8.4.126 opencv-python pandas matplotlib

In [ ]:
from pathlib import Path
import re

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ultralytics import YOLO

PATH_TO_MODEL = Path("path/to/your/best.pt")
PATH_TO_IMAGE_SEQUENCE = Path("path/to/your/image_sequence")
PATH_TO_OUTPUT_CSV = Path("tracker_measurements.csv")

MAX_SIDE = 680
CONFIDENCE = 0.25

## 2. Load the model and sequence

Natural sorting keeps numbered filenames in acquisition order (for example, `frame_2` before `frame_10`). The input sequence should contain one spheroid time series; all frames must resolve to the same dimensions after preprocessing so that pixel masks are spatially comparable.

In [ ]:
if not PATH_TO_MODEL.is_file():
    raise FileNotFoundError(f"Model weights not found: {PATH_TO_MODEL}")
if not PATH_TO_IMAGE_SEQUENCE.is_dir():
    raise NotADirectoryError(f"Image sequence directory not found: {PATH_TO_IMAGE_SEQUENCE}")

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

def natural_sort_key(path):
    return tuple(
        (part.isdigit(), int(part) if part.isdigit() else part.casefold())
        for part in re.split(r"(\d+)", path.name)
    )

frame_paths = sorted(
    (path for path in PATH_TO_IMAGE_SEQUENCE.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS),
    key=natural_sort_key,
)
if not frame_paths:
    raise FileNotFoundError(f"No supported image files found in {PATH_TO_IMAGE_SEQUENCE}")

model = YOLO(str(PATH_TO_MODEL))
print(f"Loaded {len(frame_paths)} frames and model: {PATH_TO_MODEL}")

## 3. Preprocess and segment

The method chapter specifies bilateral filtering, CLAHE, and aspect-preserving resizing to 680 pixels on the longer side. The bilateral-filter and CLAHE parameters below are explicit starting values because the paper does not report their settings; validate them on representative images before quantitative use. `retina_masks=True` requests masks at the input-frame resolution, which is required for pixel-wise IoU and area calculations.

In [ ]:
def preprocess_frame(image_path):
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    denoised = cv2.bilateralFilter(image, d=9, sigmaColor=75, sigmaSpace=75)
    lab = cv2.cvtColor(denoised, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    height, width = enhanced.shape[:2]
    scale = MAX_SIDE / max(height, width)
    new_size = (max(1, round(width * scale)), max(1, round(height * scale)))
    interpolation = cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR
    return cv2.resize(enhanced, new_size, interpolation=interpolation)

def segment_frame(image):
    result = model.predict(
        source=image,
        imgsz=MAX_SIDE,
        conf=CONFIDENCE,
        retina_masks=True,
        verbose=False,
    )[0]

    if result.masks is None:
        empty_masks = np.empty((0, image.shape[0], image.shape[1]), dtype=bool)
        return empty_masks, np.empty(0, dtype=float)

    masks = result.masks.data.detach().cpu().numpy().astype(bool)
    if masks.shape[1:] != image.shape[:2]:
        raise ValueError("Prediction masks do not match the preprocessed frame dimensions.")
    confidences = result.boxes.conf.detach().cpu().numpy()
    return masks, confidences

## 4. Associate detections with reference instances

For each first-frame reference mask, the paper selects the current-frame mask with maximum IoU. If several references select the same current mask, only the reference with the largest IoU is retained. The remaining detections have no track ID. No minimum IoU threshold is specified in the paper; accordingly, this implementation follows the argmax rule without adding one.

In [ ]:
def mask_iou_matrix(reference_masks, current_masks):
    iou = np.zeros((len(current_masks), len(reference_masks)), dtype=float)
    for current_index, current_mask in enumerate(current_masks):
        for reference_index, reference_mask in enumerate(reference_masks):
            intersection = np.count_nonzero(current_mask & reference_mask)
            union = np.count_nonzero(current_mask | reference_mask)
            iou[current_index, reference_index] = intersection / union if union else 0.0
    return iou

def match_to_reference(reference_masks, current_masks):
    current_track_ids = np.full(len(current_masks), -1, dtype=int)
    if len(reference_masks) == 0 or len(current_masks) == 0:
        return current_track_ids

    iou = mask_iou_matrix(reference_masks, current_masks)
    best_current = np.argmax(iou, axis=0)
    best_iou = iou[best_current, np.arange(len(reference_masks))]

    for current_index in np.unique(best_current):
        candidate_references = np.flatnonzero(best_current == current_index)
        selected_reference = candidate_references[np.argmax(best_iou[candidate_references])]
        current_track_ids[current_index] = selected_reference

    return current_track_ids

## 5. Process the sequence

Track IDs correspond to detection indices in the first frame. Matching always uses the first-frame masks, rather than the preceding frame; thus a missing detection does not erase the reference identity. Area is the number of foreground pixels. Equivalent diameter assumes a circle with that area, while the sphere-equivalent volume assumes a sphere with that diameter and should be interpreted only as a geometric estimate.

In [ ]:
def draw_tracks(image, masks, track_ids, confidences):
    overlay = image.copy()
    for detection_index, (mask, track_id) in enumerate(zip(masks, track_ids)):
        if track_id < 0:
            continue
        hue = (int(track_id) * 37) % 180
        color = cv2.cvtColor(
            np.uint8([[[hue, 220, 255]]]), cv2.COLOR_HSV2BGR
        )[0, 0].astype(float)
        overlay[mask] = (0.55 * overlay[mask] + 0.45 * color).astype(np.uint8)
        coordinates = np.argwhere(mask)
        if coordinates.size:
            center_y, center_x = np.round(coordinates.mean(axis=0)).astype(int)
            label = f"ID {track_id} | {confidences[detection_index]:.2f}"
            cv2.putText(overlay, label, (center_x, center_y), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color.tolist(), 1)
    return overlay

records = []
preview_overlays = {}
preview_indices = {0, len(frame_paths) // 2, len(frame_paths) - 1}
reference_masks = None
reference_shape = None

for frame_index, image_path in enumerate(frame_paths):
    frame = preprocess_frame(image_path)
    masks, confidences = segment_frame(frame)

    if frame_index == 0:
        reference_masks = masks.copy()
        reference_shape = frame.shape[:2]
        if len(reference_masks) == 0:
            raise ValueError("No spheroids were detected in the first frame; reference tracks cannot be initialized.")
        track_ids = np.arange(len(masks), dtype=int)
    else:
        if frame.shape[:2] != reference_shape:
            raise ValueError("Frames differ in size after preprocessing; pixel-mask IoU requires a common image grid.")
        track_ids = match_to_reference(reference_masks, masks)

    for detection_index, (mask, track_id) in enumerate(zip(masks, track_ids)):
        if track_id < 0:
            continue
        area_px2 = int(mask.sum())
        diameter_px = 2 * np.sqrt(area_px2 / np.pi)
        volume_px3 = (np.pi / 6) * diameter_px**3
        records.append(
            {
                "frame_index": frame_index,
                "frame_name": image_path.name,
                "track_id": int(track_id),
                "confidence": float(confidences[detection_index]),
                "area_px2": area_px2,
                "equivalent_diameter_px": diameter_px,
                "sphere_equivalent_volume_px3": volume_px3,
            }
        )

    if frame_index in preview_indices:
        preview_overlays[frame_index] = draw_tracks(frame, masks, track_ids, confidences)

measurements = pd.DataFrame.from_records(records)
if measurements.empty:
    raise ValueError("No tracked spheroid measurements were produced. Check the model, confidence, and first frame.")
measurements.to_csv(PATH_TO_OUTPUT_CSV, index=False)
print(f"Saved {len(measurements)} measurements to {PATH_TO_OUTPUT_CSV}")
display(measurements.head())

## 6. Inspect segmentation and morphology

The overlays show predicted masks and the assigned reference ID with detection confidence. The time series are descriptive outputs of inference; they are not substitutes for mask-level or tracking evaluation against annotated ground truth.

In [ ]:
fig, axes = plt.subplots(1, len(preview_overlays), figsize=(5 * len(preview_overlays), 5))
axes = np.atleast_1d(axes)
for axis, (frame_index, overlay) in zip(axes, sorted(preview_overlays.items())):
    axis.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    axis.set_title(f"Frame {frame_index}")
    axis.axis("off")
plt.tight_layout()
plt.show()

display(
    measurements.groupby("track_id")["frame_index"]
    .agg(first_frame="min", last_frame="max", observations="count")
)

plot_columns = [
    ("equivalent_diameter_px", "Equivalent diameter (px)"),
    ("area_px2", "Mask area (px^2)"),
    ("sphere_equivalent_volume_px3", "Sphere-equivalent volume estimate (px^3)"),
]
fig, axes = plt.subplots(1, len(plot_columns), figsize=(15, 4))
for axis, (column, title) in zip(axes, plot_columns):
    for track_id, group in measurements.groupby("track_id", sort=True):
        axis.plot(group["frame_index"], group[column], marker=".", linewidth=1, label=f"ID {track_id}")
    axis.set_title(title)
    axis.set_xlabel("Frame index")
    axis.grid(alpha=0.25)
axes[0].set_ylabel("Measurement")
axes[-1].legend(title="Track", bbox_to_anchor=(1.04, 1), loc="upper left")
plt.tight_layout()
plt.show()